# LIVE fMRI Visual Reconstruction — Subject-Agnostic CLIP–DINOv2 Guided Diffusion

This notebook is the **LIVE-specific counterpart of the supplied Haxby reconstruction notebook**. It keeps the same Transformer → CLIP/DINOv2 → diffusion-prior → retrieval → Stable Diffusion Img2Img flow and the same evaluation family, while adapting the input contract to the LIVE dataset described in `vgst1.pdf`.

**LIVE methodology alignment:** six subjects; S1–S4 training, S5 validation, S6 held-out testing; 4D BOLD volumes of **147 × 144 × 36 × 165**, TR **3.0 s**; natural-scene stimuli linked to COCO; VT ROI extraction, detrending, band-pass filtering, and voxel-wise standardization before shared latent alignment. fileciteturn1file0L229-L258

The uploaded Haxby notebook does not contain the original LIVE acquisition/preprocessing code, so this notebook keeps the raw-BOLD preprocessing explicit and requires the final shared-aligned `[T,128]`/`[N,20,128]` representation for the supplied decoder architecture rather than silently inventing a registration/alignment method.

In [ ]:
!pip install -q open_clip_torch diffusers transformers timm h5py scikit-image scipy pandas scikit-learn

In [ ]:
# =========================================================
# LIVE CONFIGURATION
# =========================================================
from pathlib import Path

LIVE_ROOT = Path("/content/live")              # change if needed
PREPARED_DIR = LIVE_ROOT / "prepared"
CHECKPOINT_DIR = LIVE_ROOT / "checkpoints"
OUTPUT_DIR = LIVE_ROOT / "reconstructions"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_ALIGNED_PATH = PREPARED_DIR / "test_aligned.npy"
TRAIN_IMAGE_MANIFEST = PREPARED_DIR / "train_image_paths.npy"
TEST_IMAGE_MANIFEST = PREPARED_DIR / "test_image_paths.npy"
RETRIEVAL_CACHE = PREPARED_DIR / "train_retrieval_db.pt"

TRAIN_SUBJECTS = ["S1","S2","S3","S4"]
VAL_SUBJECTS = ["S5"]
TEST_SUBJECTS = ["S6"]

MAX_RECONSTRUCTIONS = 24

# Methodology acquisition values for validation/documentation.
BOLD_SHAPE = (147, 144, 36, 165)
TR_SECONDS = 3.0

print("LIVE root:", LIVE_ROOT)
print("Expected 4D BOLD:", BOLD_SHAPE)
print("TR:", TR_SECONDS, "seconds")
print("Train:", TRAIN_SUBJECTS, "Validation:", VAL_SUBJECTS, "Test:", TEST_SUBJECTS)

In [ ]:
import os, math, csv, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from PIL import Image, ImageFilter
from torchvision import transforms, models
from torchvision.models.feature_extraction import create_feature_extractor
from scipy.stats import pearsonr, ttest_rel, wilcoxon
from skimage.metrics import mean_squared_error, peak_signal_noise_ratio, structural_similarity
import open_clip

from diffusers import StableDiffusionImg2ImgPipeline

warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
print("Using device:", DEVICE)

## 1. LIVE BOLD preprocessing contract

The methodology specifies: **VT ROI extraction → detrending → band-pass filtering → voxel-wise standardization → shared latent alignment**. The raw acquisition is 147×144×36×165 with TR=3.0 s. The cell below is an explicit preprocessing utility for a single 4D BOLD NIfTI; it does not claim a particular VT mask that is not provided in the uploaded files. Supply the actual VT ROI mask used by your study.

After this step, use the same shared-alignment stage that produced the dataset-specific `test_aligned.npy` checkpoint input.

In [ ]:
import nibabel as nib
from scipy import signal
from sklearn.preprocessing import StandardScaler

def load_bold_nifti(path):
    img = nib.load(str(path))
    data = img.get_fdata().astype(np.float32)
    print("BOLD shape:", data.shape)
    if tuple(data.shape) != BOLD_SHAPE:
        print("Warning: shape differs from the methodology-reported", BOLD_SHAPE)
    return data

def preprocess_live_bold(bold_4d, vt_mask, tr=TR_SECONDS, low_hz=0.01, high_hz=0.1):
    """
    Methodology-aligned preprocessing:
      1) VT ROI extraction
      2) detrending
      3) band-pass filtering
      4) voxel-wise standardization

    Input: 4D array [X,Y,Z,T], boolean VT mask [X,Y,Z].
    Output: [T,V] standardized ROI responses.
    """
    if bold_4d.ndim != 4:
        raise ValueError("Expected 4D BOLD [X,Y,Z,T].")
    if vt_mask.shape != bold_4d.shape[:3]:
        raise ValueError("VT mask shape must match the first three BOLD dimensions.")

    roi = bold_4d[vt_mask].T  # [T,V]
    roi = signal.detrend(roi, axis=0, type="linear")

    nyquist = 0.5 / tr
    b, a = signal.butter(
        2,
        [low_hz/nyquist, high_hz/nyquist],
        btype="bandpass"
    )
    roi = signal.filtfilt(b, a, roi, axis=0)

    roi = StandardScaler().fit_transform(roi).astype(np.float32)
    return roi

print("LIVE preprocessing utility defined.")

In [ ]:
# =========================================================
# LOAD SHARED-ALIGNED LIVE TEST DATA
# =========================================================
def load_image_paths(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Image-path manifest not found: {path}")
    if path.suffix.lower() == ".npy":
        return [str(x) for x in np.load(path, allow_pickle=True).tolist()]
    if path.suffix.lower() in [".txt",".lst"]:
        return [x.strip() for x in path.read_text().splitlines() if x.strip()]
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
        col = "image_path" if "image_path" in df.columns else df.columns[0]
        return df[col].astype(str).tolist()
    raise ValueError("Use .npy, .txt, .lst or .csv manifests.")

if not TEST_ALIGNED_PATH.exists():
    raise FileNotFoundError(
        f"{TEST_ALIGNED_PATH} not found. Run the LIVE preprocessing/alignment pipeline first."
    )

X = np.load(TEST_ALIGNED_PATH, allow_pickle=True)
train_image_paths = load_image_paths(TRAIN_IMAGE_MANIFEST)
test_image_paths = load_image_paths(TEST_IMAGE_MANIFEST)

print("Aligned LIVE test shape:", X.shape)
print("Train images:", len(train_image_paths))
print("Test images:", len(test_image_paths))

In [ ]:

INPUT_DIM = 128
LATENT_DIM = 256
SEQ_LEN = 20
CLIP_WEIGHT = 0.35
DINO_WEIGHT = 0.65

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) *
            (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class FMRITransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Linear(INPUT_DIM, LATENT_DIM)
        self.pos = PositionalEncoding(LATENT_DIM)
        self.cls = nn.Parameter(torch.randn(1, 1, LATENT_DIM))
        layer = nn.TransformerEncoderLayer(
            d_model=LATENT_DIM, nhead=8, dim_feedforward=512,
            dropout=0.1, activation="gelu", batch_first=True
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=4)
        self.attention_pool = nn.Sequential(
            nn.Linear(LATENT_DIM, 128), nn.Tanh(), nn.Linear(128, 1)
        )
        self.norm = nn.LayerNorm(LATENT_DIM)
        self.head = nn.Sequential(
            nn.Linear(LATENT_DIM, 512), nn.GELU(), nn.Linear(512, INPUT_DIM)
        )

    def forward(self, x):
        B = x.shape[0]
        x = self.embedding(x)
        cls = self.cls.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = self.pos(x)
        x = self.transformer(x)
        attn = torch.softmax(self.attention_pool(x), dim=1)
        pooled = (x * attn).sum(dim=1)
        latent = self.norm(pooled)
        recon = self.head(latent)
        return latent, recon

class FMRIToCLIP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(256, 1024), nn.LayerNorm(1024), nn.GELU(),
            nn.Dropout(0.2), nn.Linear(1024, 1024), nn.LayerNorm(1024),
            nn.GELU(), nn.Dropout(0.2), nn.Linear(1024, 512)
        )
    def forward(self, x): return self.net(x)

class FMRIToDINO(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(256, 1024), nn.LayerNorm(1024), nn.GELU(),
            nn.Dropout(0.2), nn.Linear(1024, 768)
        )
    def forward(self, x): return self.net(x)

class DiffusionPrior(nn.Module):
    def __init__(self):
        super().__init__()
        self.time_embed = nn.Sequential(
            nn.Linear(1, 512), nn.GELU(), nn.Linear(512, 512)
        )
        self.net = nn.Sequential(
            nn.Linear(1024, 2048), nn.LayerNorm(2048),
            nn.GELU(), nn.Linear(2048, 512)
        )
    def forward(self, x, t):
        t_embed = self.time_embed(t)
        return self.net(torch.cat([x, t_embed], dim=-1))

fmri_model = FMRITransformer().to(DEVICE)
clip_projector = FMRIToCLIP().to(DEVICE)
dino_projector = FMRIToDINO().to(DEVICE)
prior = DiffusionPrior().to(DEVICE)

def load_checkpoint(module, path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing checkpoint: {path}. "
            "These checkpoints must be trained for this dataset; do not reuse Haxby weights unless they were trained on this dataset."
        )
    module.load_state_dict(torch.load(path, map_location=DEVICE))
    module.eval()
    print("Loaded:", path)

load_checkpoint(fmri_model, CHECKPOINT_DIR / "transformer_best.pth")
load_checkpoint(clip_projector, CHECKPOINT_DIR / "clip_projector_best.pth")
load_checkpoint(dino_projector, CHECKPOINT_DIR / "dino_projector_best.pth")
load_checkpoint(prior, CHECKPOINT_DIR / "diffusion_prior_best.pth")


# Optional second prior required by the paper's dual-prior formulation.
# The supplied Haxby notebook contains only diffusion_prior_best.pth, so this
# branch is enabled only when a dataset-specific DINO prior checkpoint exists.
class DINOFeaturePrior(nn.Module):
    def __init__(self):
        super().__init__()
        self.time_embed = nn.Sequential(
            nn.Linear(1, 512), nn.GELU(), nn.Linear(512, 512)
        )
        self.net = nn.Sequential(
            nn.Linear(768 + 512, 2048), nn.LayerNorm(2048),
            nn.GELU(), nn.Linear(2048, 768)
        )
    def forward(self, x, t):
        t_embed = self.time_embed(t)
        return self.net(torch.cat([x, t_embed], dim=-1))

dino_prior = None
dino_prior_path = CHECKPOINT_DIR / "dino_diffusion_prior_best.pth"
if dino_prior_path.exists():
    dino_prior = DINOFeaturePrior().to(DEVICE)
    dino_prior.load_state_dict(torch.load(dino_prior_path, map_location=DEVICE))
    dino_prior.eval()
    print("Loaded optional dual-prior DINO checkpoint:", dino_prior_path)
else:
    print("No dino_diffusion_prior_best.pth found; using the supplied Haxby notebook's single-prior DINO branch.")


In [ ]:
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
clip_model = clip_model.to(DEVICE).float().eval()

dino_preprocess = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485,0.456,0.406),
        std=(0.229,0.224,0.225)
    )
])

dinov2 = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14")
dinov2 = dinov2.to(DEVICE).eval()

from fmri_reconstruction.reconstruction.stable_diffusion import StableDiffusionImg2Img

pipe = StableDiffusionImg2Img(
    model_id="runwayml/stable-diffusion-v1-5",
    device=DEVICE,
    torch_dtype=DTYPE,
    disable_safety=True,
    enable_xformers=False,
)

print("CLIP, DINOv2 and Stable Diffusion loaded.")

In [ ]:
def load_image_paths(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Image-path manifest not found: {path}")
    if path.suffix.lower() == ".npy":
        arr = np.load(path, allow_pickle=True)
        return [str(x) for x in arr.tolist()]
    if path.suffix.lower() in [".txt", ".lst"]:
        return [x.strip() for x in path.read_text().splitlines() if x.strip()]
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
        col = "image_path" if "image_path" in df.columns else df.columns[0]
        return df[col].astype(str).tolist()
    raise ValueError("Use .npy, .txt, .lst or .csv for image-path manifests.")

def encode_image_database(image_paths, cache_path):
    cache_path = Path(cache_path)
    if cache_path.exists():
        print("Loading cached retrieval database:", cache_path)
        return torch.load(cache_path, map_location=DEVICE)

    clip_list, dino_list, valid_paths = [], [], []
    for i, path in enumerate(image_paths):
        if not Path(path).exists():
            continue
        img = Image.open(path).convert("RGB")
        ct = clip_preprocess(img).unsqueeze(0).to(DEVICE)
        dt = dino_preprocess(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            c = F.normalize(clip_model.encode_image(ct).float(), dim=-1)
            try:
                feats = dinov2.forward_features(dt)
                d = feats["x_norm_clstoken"]
            except Exception:
                d = dinov2(dt)
            d = F.normalize(d.float(), dim=-1)
        clip_list.append(c.squeeze(0).cpu())
        dino_list.append(d.squeeze(0).cpu())
        valid_paths.append(path)
        if (i + 1) % 500 == 0:
            print(f"Encoded {i+1}/{len(image_paths)} images")

    if not valid_paths:
        raise RuntimeError("No valid training images were found.")

    db = {
        "clip": torch.stack(clip_list).float().to(DEVICE),
        "dino": torch.stack(dino_list).float().to(DEVICE),
        "images": valid_paths
    }
    torch.save(db, cache_path)
    print("Saved retrieval database:", cache_path)
    return db

database = encode_image_database(TRAIN_IMAGE_MANIFEST, RETRIEVAL_CACHE)

def retrieve_best_image(pred_clip, pred_dino, top_k=30):
    with torch.no_grad():
        clip_sim = pred_clip @ database["clip"].T
        dino_sim = pred_dino @ database["dino"].T
        fusion = CLIP_WEIGHT * clip_sim + DINO_WEIGHT * dino_sim
        k = min(top_k, fusion.shape[1])
        idxs = fusion.topk(k, dim=1).indices[0]
        best = idxs[0].item()
    return (
        database["images"][best],
        clip_sim[0, best].item(),
        dino_sim[0, best].item(),
        fusion[0, best].item()
    )

def get_prompt():
    return "high quality realistic natural scene photograph, detailed visual structure"

def get_adaptive_strength(sim):
    if sim > 0.92: return 0.08
    if sim > 0.85: return 0.12
    if sim > 0.75: return 0.16
    return 0.22

def get_adaptive_guidance(clip_score):
    if clip_score > 0.90: return 4.5
    if clip_score > 0.80: return 5.5
    if clip_score > 0.70: return 6.5
    return 8.0

In [ ]:

def prepare_sequences(X):
    X = np.asarray(X, dtype=np.float32)
    if X.ndim == 3:
        if X.shape[1:] != (SEQ_LEN, INPUT_DIM):
            raise ValueError(f"Expected [N,{SEQ_LEN},{INPUT_DIM}], got {X.shape}")
        return [(X[i], i) for i in range(len(X))]
    if X.ndim == 2:
        if X.shape[1] != INPUT_DIM:
            raise ValueError(f"Expected [T,{INPUT_DIM}], got {X.shape}")
        if len(X) < SEQ_LEN:
            return []
        return [(X[i:i+SEQ_LEN], i+SEQ_LEN-1) for i in range(len(X)-SEQ_LEN+1)]
    raise ValueError("Aligned fMRI must be 2-D [T,128] or 3-D [N,20,128].")

def generate_reconstructions(X, test_image_paths, max_samples=24):
    seqs = prepare_sequences(X)
    if len(test_image_paths) != (len(X) if X.ndim == 3 else len(X)):
        print("Warning: manifest length does not equal raw aligned length.")
    reconstructions = []
    n = min(len(seqs), max_samples) if max_samples is not None else len(seqs)

    for j in range(n):
        x_seq, img_idx = seqs[j]
        if img_idx >= len(test_image_paths):
            continue
        original_path = test_image_paths[img_idx]
        if not Path(original_path).exists():
            continue

        original = Image.open(original_path).convert("RGB")
        x_tensor = torch.tensor(x_seq, dtype=torch.float32).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            latent, _ = fmri_model(x_tensor)
            pred_clip = F.normalize(
                prior(F.normalize(clip_projector(latent), dim=-1),
                      torch.zeros(1,1,device=DEVICE)), dim=-1
            )
            pred_dino = F.normalize(dino_projector(latent), dim=-1)
            if dino_prior is not None:
                pred_dino = F.normalize(
                    dino_prior(pred_dino, torch.zeros(1,1,device=DEVICE)),
                    dim=-1
                )

        retrieved_path, clip_score, dino_score, fusion_score = retrieve_best_image(
            pred_clip, pred_dino
        )
        retrieved = Image.open(retrieved_path).convert("RGB")

        strength = get_adaptive_strength(fusion_score)
        guidance = get_adaptive_guidance(clip_score)

        with torch.autocast("cuda", enabled=(DEVICE == "cuda")):
            generated = pipe(
                prompt=get_prompt(),
                image=retrieved.resize((512,512)),
                strength=strength,
                guidance_scale=guidance,
                num_inference_steps=100
            ).images[0]

        out_path = OUTPUT_DIR / f"reconstruction_{j:05d}.png"
        generated.save(out_path)
        reconstructions.append({
            "index": j,
            "original_path": original_path,
            "retrieved_path": retrieved_path,
            "generated_path": str(out_path),
            "clip_score": clip_score,
            "dino_score": dino_score,
            "fusion_score": fusion_score
        })
        print(f"{j+1}/{n}: saved {out_path}")

    pd.DataFrame(reconstructions).to_csv(OUTPUT_DIR / "reconstruction_log.csv", index=False)
    return reconstructions


In [ ]:
# =========================================================
# RUN RECONSTRUCTION
# =========================================================
records = generate_reconstructions(
    X,
    test_image_paths,
    max_samples=MAX_RECONSTRUCTIONS
)
print("Generated:", len(records), "reconstructions")

In [ ]:
# =========================================================
# VISUALIZE ORIGINAL / RETRIEVED / RECONSTRUCTED
# =========================================================
log_df = pd.read_csv(OUTPUT_DIR / "reconstruction_log.csv")
show_n = min(6, len(log_df))

fig, axes = plt.subplots(3, show_n, figsize=(4*show_n, 12))
if show_n == 1:
    axes = np.expand_dims(axes, axis=1)

for col, (_, row) in enumerate(log_df.head(show_n).iterrows()):
    axes[0,col].imshow(Image.open(row["original_path"]).convert("RGB"))
    axes[0,col].set_title("Original")
    axes[0,col].axis("off")

    axes[1,col].imshow(Image.open(row["retrieved_path"]).convert("RGB"))
    axes[1,col].set_title(f"Retrieved\nFusion={row['fusion_score']:.2f}")
    axes[1,col].axis("off")

    axes[2,col].imshow(Image.open(row["generated_path"]).convert("RGB"))
    axes[2,col].set_title(f"Reconstructed\nCLIP={row['clip_score']:.2f}")
    axes[2,col].axis("off")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "live_qualitative_results.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
alex = models.alexnet(weights=models.AlexNet_Weights.DEFAULT).to(DEVICE).eval()
inception = models.inception_v3(weights=models.Inception_V3_Weights.DEFAULT).to(DEVICE).eval()
efficient = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT).to(DEVICE).eval()
try:
    swav = torch.hub.load("facebookresearch/swav:main", "resnet50").to(DEVICE).eval()
except Exception:
    swav = models.resnet50(weights=models.ResNet50_Weights.DEFAULT).to(DEVICE).eval()

alex_ex = create_feature_extractor(alex, return_nodes={"features.3":"alex2","features.10":"alex5"})
efficient_ex = create_feature_extractor(efficient, return_nodes={"features":"efficient"})
swav_ex = create_feature_extractor(swav, return_nodes={"avgpool":"swav"})

transform_224 = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor()])
transform_299 = transforms.Compose([transforms.Resize(320), transforms.CenterCrop(299), transforms.ToTensor()])

def load_tensor(path, size):
    img = Image.open(path).convert("RGB")
    tensor = transform_299(img) if size == 299 else transform_224(img)
    return img, tensor.unsqueeze(0)

def feat_to_vector(x):
    if x.dim() == 4:
        x = F.adaptive_avg_pool2d(x, 1)
    return x.flatten(1)

def cosine(a,b):
    return float(F.cosine_similarity(a,b).item())

def evaluate_outputs(log_df):
    rows = []
    for _, r in log_df.iterrows():
        op, gp = r["original_path"], r["generated_path"]
        if not Path(op).exists() or not Path(gp).exists():
            continue
        orig, o224 = load_tensor(op, 224)
        gen, g224 = load_tensor(gp, 224)
        _, o299 = load_tensor(op, 299)
        _, g299 = load_tensor(gp, 299)

        on = np.array(orig.resize((256,256)))
        gn = np.array(gen.resize((256,256)))
        mse = mean_squared_error(on, gn)
        psnr = peak_signal_noise_ratio(on, gn, data_range=255)
        ssim = structural_similarity(on, gn, channel_axis=2, data_range=255)
        pixcorr = pearsonr(on.flatten(), gn.flatten())[0]

        with torch.no_grad():
            ao, ag = alex_ex(o224.to(DEVICE)), alex_ex(g224.to(DEVICE))
            alex2 = cosine(feat_to_vector(ao["alex2"]), feat_to_vector(ag["alex2"]))
            alex5 = cosine(feat_to_vector(ao["alex5"]), feat_to_vector(ag["alex5"]))

            io, ig = inception(o299.to(DEVICE)), inception(g299.to(DEVICE))
            if hasattr(io, "logits"): io = io.logits
            if hasattr(ig, "logits"): ig = ig.logits
            inception_sim = cosine(io, ig)

            eo, eg = efficient_ex(o224.to(DEVICE)), efficient_ex(g224.to(DEVICE))
            efficient_sim = cosine(feat_to_vector(eo["efficient"]), feat_to_vector(eg["efficient"]))

            so, sg = swav_ex(o224.to(DEVICE)), swav_ex(g224.to(DEVICE))
            swav_sim = cosine(feat_to_vector(so["swav"]), feat_to_vector(sg["swav"]))

            co = clip_preprocess(orig).unsqueeze(0).to(DEVICE)
            cg = clip_preprocess(gen).unsqueeze(0).to(DEVICE)
            emb_o = F.normalize(clip_model.encode_image(co).float(), dim=-1)
            emb_g = F.normalize(clip_model.encode_image(cg).float(), dim=-1)
            clip_sim = float(F.cosine_similarity(emb_o, emb_g).item())

        rows.append([int(r["index"]), mse, psnr, ssim, pixcorr, alex2, alex5,
                     inception_sim, efficient_sim, swav_sim, clip_sim])

    cols = ["sample_id","mse","psnr","ssim","pixcorr","alex2","alex5",
            "inception","efficientnet","swav","clip"]
    df = pd.DataFrame(rows, columns=cols)
    df.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
    print(df)
    if len(df):
        print("\nAverage metrics:")
        print(df.drop(columns=["sample_id"]).mean(numeric_only=True).round(4))
    return df

In [ ]:
# =========================================================
# EVALUATE LIVE RECONSTRUCTIONS
# =========================================================
metrics_df = evaluate_outputs(log_df)

In [ ]:
# CLIP-only retrieval ablation, matching the Haxby notebook's ablation logic.
def retrieve_clip_only(pred_clip):
    with torch.no_grad():
        sim = pred_clip @ database["clip"].T
        idx = sim.argmax(dim=1).item()
    return database["images"][idx], sim[0,idx].item()

def run_clip_retrieval_ablation(X, test_image_paths, max_samples=6):
    seqs = prepare_sequences(X)
    rows = []
    n = min(len(seqs), max_samples) if max_samples is not None else len(seqs)
    for j in range(n):
        x_seq, img_idx = seqs[j]
        if img_idx >= len(test_image_paths): continue
        original_path = test_image_paths[img_idx]
        if not Path(original_path).exists(): continue
        x_tensor = torch.tensor(x_seq, dtype=torch.float32).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            latent, _ = fmri_model(x_tensor)
            pred_clip = F.normalize(
                prior(F.normalize(clip_projector(latent), dim=-1),
                      torch.zeros(1,1,device=DEVICE)), dim=-1
            )
        retrieved_path, score = retrieve_clip_only(pred_clip)
        retrieved = Image.open(retrieved_path).convert("RGB")
        with torch.autocast("cuda", enabled=(DEVICE=="cuda")):
            generated = pipe(
                prompt=get_prompt(),
                image=retrieved.resize((512,512)),
                strength=0.22,
                guidance_scale=8.0,
                num_inference_steps=100
            ).images[0]
        out = OUTPUT_DIR / f"ablation_clip_retrieval_{j:05d}.png"
        generated.save(out)
        rows.append([j, original_path, retrieved_path, str(out), score])
    pd.DataFrame(rows, columns=["sample_id","original_path","retrieved_path","generated_path","clip_score"]).to_csv(
        OUTPUT_DIR / "ablation_clip_retrieval_log.csv", index=False
    )
    return rows

In [ ]:
# =========================================================
# CLIP + RETRIEVAL ABLATION
# =========================================================
ablation_records = run_clip_retrieval_ablation(
    X, test_image_paths, max_samples=min(6, MAX_RECONSTRUCTIONS or 6)
)
print("Ablation outputs:", len(ablation_records))

### LIVE notes

- The methodology reports six subjects with **S1–S4 training, S5 validation, S6 testing**, 4D BOLD size **147×144×36×165**, and TR **3.0 s**. fileciteturn1file0L251-L258
- The reported LIVE full-model reference average is **SSIM 0.3917, PixCorr 0.3707, AlexNet(2) 77.56%, AlexNet(5) 82.60%, Inception 71.52%, CLIP 74.16%, EffNet-B 0.6654, SwAV 0.5965**. The notebook computes metrics from your generated images rather than inserting those numbers. fileciteturn1file1L83-L100
- The methodology reports a retrieval optimum of **82.15%** for LIVE at Top-25. This is a reported paper value, not a value hard-coded into the notebook. fileciteturn1file1L96-L106
- Use **training images only** in the retrieval manifest to avoid test leakage.